In [ ]:
%pip install torch ta mplfinance scikit-learn matplotlib pandas

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import pandas as pd
import ta

SYMBOL = 'XRPUSDT'
MODEL_TYPE = 'lstm'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    print("Not running in Google Colab. Using local directory.")
    BASE_DIR = os.path.abspath(os.getcwd())

DATA_DIR = os.path.join(BASE_DIR, f'processed_data_{MODEL_TYPE}', SYMBOL)
MODEL_DIR = os.path.join(BASE_DIR, 'models')
CSV_PATH = os.path.join(BASE_DIR, 'data', f'{SYMBOL}_5m_data.csv')
MODEL_SAVE_PATH = os.path.join(MODEL_DIR, f'best_{MODEL_TYPE}_vol_model_{SYMBOL}.pth')
DASHBOARD_FILE_PATH = os.path.join(BASE_DIR, f'{MODEL_TYPE}_vol_dashboard_data_{SYMBOL}.csv')
os.makedirs(MODEL_DIR, exist_ok=True)

# --- Hyperparameters ---
BATCH_SIZE = 64
EPOCHS = 100             # Max ceiling - EarlyStopping ends training well before this
LEARNING_RATE = 0.001
HIDDEN_DIM = 64          # Smaller model - less prone to memorising (was 128)
NUM_LAYERS = 2
DROPOUT = 0.4            # Stronger regularisation against overfitting (was 0.2)
WEIGHT_DECAY = 1e-3      # Stronger L2 penalty against overfitting (was 1e-4)
EARLY_STOP_PATIENCE = 5  # Stop soon after the val minimum - keeps the best model
SEQ_LENGTH = 60          # Must match preprocessing
VOL_WINDOW = 12          # Must match preprocessing
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1
EPS = 1e-8               # Small constant for the log-volatility transform


class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.hidden_dim = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.dropout(out[:, -1, :])
        return self.fc(out)


class EarlyStopping:
    def __init__(self, patience=12, delta=1e-6):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_loss = np.inf

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self._save(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'EarlyStopping: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self._save(val_loss, model)
            self.counter = 0

    def _save(self, val_loss, model):
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        self.best_loss = val_loss


def load_data():
    print(f"Loading preprocessed data from {DATA_DIR}...")
    X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
    y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
    X_val   = np.load(os.path.join(DATA_DIR, 'X_val.npy'))
    y_val   = np.load(os.path.join(DATA_DIR, 'y_val.npy'))
    X_test  = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
    y_test  = np.load(os.path.join(DATA_DIR, 'y_test.npy'))
    # Persistence baseline (current realized vol) aligned with each target
    b_test  = np.load(os.path.join(DATA_DIR, 'b_test.npy'))
    print(f"  Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
    print(f"  Target (future vol) std (train): {y_train.std():.6f}")
    return X_train, y_train, X_val, y_val, X_test, y_test, b_test


def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    X_train, y_train, X_val, y_val, X_test, y_test, b_test = load_data()

    # Features are already StandardScaled from preprocessing - do NOT scale again.
    #
    # TARGET = LOG of future volatility, then StandardScaled.
    # Volatility is right-skewed (rare large spikes dominate). Taking log() makes the
    # distribution roughly symmetric, so training is more stable across calm/volatile
    # regimes. We convert predictions back with exp() before evaluating, so all the
    # reported metrics stay in real volatility units (comparable to the baseline).
    y_train_log = np.log(y_train + EPS)
    y_val_log   = np.log(y_val + EPS)

    y_scaler = StandardScaler()
    y_train_s = y_scaler.fit_transform(y_train_log.reshape(-1, 1)).flatten()
    y_val_s   = y_scaler.transform(y_val_log.reshape(-1, 1)).flatten()

    X_train_t = torch.tensor(X_train,   dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train_s, dtype=torch.float32).to(device)
    X_val_t   = torch.tensor(X_val,     dtype=torch.float32).to(device)
    y_val_t   = torch.tensor(y_val_s,   dtype=torch.float32).to(device)
    X_test_t  = torch.tensor(X_test,    dtype=torch.float32).to(device)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(TensorDataset(X_test_t,),             batch_size=BATCH_SIZE, shuffle=False)

    n_features = X_train.shape[2]
    model = LSTMModel(
        input_size=n_features, hidden_size=HIDDEN_DIM,
        num_layers=NUM_LAYERS, dropout=DROPOUT
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {n_features} features | LSTM({HIDDEN_DIM}x{NUM_LAYERS}) | Params: {n_params:,}")

    criterion  = nn.HuberLoss(delta=1.0)
    optimizer  = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-5
    )
    early_stop = EarlyStopping(patience=EARLY_STOP_PATIENCE, delta=1e-6)

    print(f"\nTraining {SYMBOL} LSTM (log-volatility)...")
    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for X_b, y_b in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_b).squeeze(), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * X_b.size(0)
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                val_loss += criterion(model(X_b).squeeze(), y_b).item() * X_b.size(0)
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)

        scheduler.step(val_loss)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_loss:.5f} | Val: {val_loss:.5f} | LR: {optimizer.param_groups[0]["lr"]:.6f}')

        early_stop(val_loss, model)
        if early_stop.early_stop:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # --- Test evaluation ---
    print("\nEvaluating on Test Set...")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    model.eval()

    preds_s = []
    with torch.no_grad():
        for (X_b,) in test_loader:
            preds_s.extend(model(X_b).squeeze().cpu().numpy())

    preds_s     = np.array(preds_s).flatten()
    # Undo StandardScaler -> undo log() -> back to real volatility units
    pred_log    = y_scaler.inverse_transform(preds_s.reshape(-1, 1)).flatten()
    predictions = np.exp(pred_log) - EPS
    predictions = np.clip(predictions, 0, None)   # volatility can't be negative

    # --- Model metrics ---
    rmse  = np.sqrt(mean_squared_error(y_test, predictions))
    mae   = mean_absolute_error(y_test, predictions)
    corr  = np.corrcoef(predictions, y_test)[0, 1]

    # --- Persistence baseline (predict future vol = current vol) ---
    base_rmse = np.sqrt(mean_squared_error(y_test, b_test))
    base_corr = np.corrcoef(b_test, y_test)[0, 1]

    improvement = (base_rmse - rmse) / base_rmse * 100

    print(f"\n--- Results (future {VOL_WINDOW}-candle realized volatility) ---")
    print(f"{'':22}{'MODEL':>12}{'PERSISTENCE':>14}")
    print(f"{'RMSE':22}{rmse:>12.6f}{base_rmse:>14.6f}")
    print(f"{'Correlation':22}{corr:>12.3f}{base_corr:>14.3f}")
    print(f"{'MAE':22}{mae:>12.6f}")
    print(f"\nRMSE improvement over persistence baseline: {improvement:+.1f}%")
    if improvement > 2:
        print(">> Model BEATS the naive baseline - it learned something real. ")
    elif improvement > -2:
        print(">> Model roughly matches persistence. Volatility is predictable, but the LSTM adds little over 'tomorrow = today'.")
    else:
        print(">> Model is worse than persistence - persistence is the better predictor here.")

    # --- Loss curve ---
    plt.figure(figsize=(10, 4))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses,   label='Val Loss')
    plt.title(f'{SYMBOL} LSTM Volatility Training Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

    # --- Prediction vs actual ---
    n_plot = min(500, len(y_test))
    plt.figure(figsize=(14, 5))
    plt.plot(y_test[:n_plot],      label='Actual Vol',    color='blue', alpha=0.7)
    plt.plot(predictions[:n_plot], label='Predicted Vol', color='red',  linewidth=1.2)
    plt.plot(b_test[:n_plot],      label='Persistence',   color='green', alpha=0.4, linewidth=0.8)
    plt.title(f'{SYMBOL} LSTM Volatility Predictions\nRMSE={rmse:.5f} | Corr={corr:.2f} | vs baseline {improvement:+.1f}%')
    plt.xlabel('Time Steps (5m)'); plt.ylabel('Realized Volatility')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

    # --- Dashboard export ---
    df_full = pd.read_csv(CSV_PATH)
    df_full['open_time'] = pd.to_datetime(df_full['open_time'])
    df_full.set_index('open_time', inplace=True)
    n_rows     = len(df_full)
    val_end    = int(n_rows * (TRAIN_SPLIT + VAL_SPLIT))
    test_start = val_end + SEQ_LENGTH
    df_test    = df_full.iloc[test_start : test_start + len(predictions)].copy()
    df_test['predicted_volatility'] = predictions
    df_test.to_csv(DASHBOARD_FILE_PATH)
    print(f"Saved dashboard to {DASHBOARD_FILE_PATH}")


if __name__ == "__main__":
    train()


In [ ]:
# ============================================================
#  PRICE PREDICTION CHART  (candlestick + predicted price + volume)
#  Uses the RETURNS model dashboard (lstm_dashboard_data_XRPUSDT.csv).
#  NOTE: the red line is a 1-step-lagging predictor - looks good, but
#  is essentially "next price = current price".
# ============================================================
%pip install mplfinance
import os
import pandas as pd
import mplfinance as mpf

SYMBOL = 'XRPUSDT'
N_PLOT = 500   # how many candles to show

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    BASE_DIR = os.path.abspath(os.getcwd())

# Dashboard produced by the RETURNS trainer (has predicted_close)
DASH_PATH = os.path.join(BASE_DIR, f'lstm_dashboard_data_{SYMBOL}.csv')

df = pd.read_csv(DASH_PATH)
df['open_time'] = pd.to_datetime(df['open_time'])
df = df.set_index('open_time')

# mplfinance needs capitalised OHLCV column names
df = df.rename(columns={'open': 'Open', 'high': 'High', 'low': 'Low',
                        'close': 'Close', 'volume': 'Volume'})

# Take the most recent N_PLOT candles
df_plot = df.iloc[:N_PLOT]

# Red dashed line = predicted price
ap = mpf.make_addplot(df_plot['predicted_close'], color='red', linestyle='--', width=1.2)

mpf.plot(
    df_plot,
    type='candle',
    style='yahoo',
    addplot=ap,
    volume=True,
    figsize=(14, 8),
    title=f'{SYMBOL} Prediction vs Actual',
    ylabel='Price (USDT)',
    ylabel_lower='Volume',
    datetime_format='%b %d, %H:%M',
)


In [ ]:
# ============================================================
#  VOLATILITY PREDICTION CHART  (the REAL working model)
#  Candlesticks on top, predicted future volatility in a panel below.
#  Uses the volatility dashboard (lstm_vol_dashboard_data_XRPUSDT.csv).
# ============================================================
%pip install mplfinance
import os
import pandas as pd
import mplfinance as mpf

SYMBOL = 'XRPUSDT'
N_PLOT = 500

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    BASE_DIR = os.path.abspath(os.getcwd())

DASH_PATH = os.path.join(BASE_DIR, f'lstm_vol_dashboard_data_{SYMBOL}.csv')

df = pd.read_csv(DASH_PATH)
df['open_time'] = pd.to_datetime(df['open_time'])
df = df.set_index('open_time')
df = df.rename(columns={'open': 'Open', 'high': 'High', 'low': 'Low',
                        'close': 'Close', 'volume': 'Volume'})

df_plot = df.iloc[:N_PLOT]

# Predicted future volatility in its own panel (panel=1)
ap = mpf.make_addplot(df_plot['predicted_volatility'], panel=1, color='red',
                      width=1.2, ylabel='Pred. Volatility')

mpf.plot(
    df_plot,
    type='candle',
    style='yahoo',
    addplot=ap,
    figsize=(14, 8),
    title=f'{SYMBOL} Price + Predicted Volatility',
    ylabel='Price (USDT)',
    datetime_format='%b %d, %H:%M',
    panel_ratios=(3, 1),
)


In [ ]:
# ============================================================
#  ACTUAL vs PREDICTED LINE CHART  (blue/red, like the ETH plot)
#  Rebuilt from the existing volatility dashboard - no retraining.
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SYMBOL = 'XRPUSDT'
VOL_WINDOW = 12   # must match preprocessing
N_PLOT = 600

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    BASE_DIR = os.path.abspath(os.getcwd())

DASH_PATH = os.path.join(BASE_DIR, f'lstm_vol_dashboard_data_{SYMBOL}.csv')

df = pd.read_csv(DASH_PATH)
df['open_time'] = pd.to_datetime(df['open_time'])
df = df.sort_values('open_time').reset_index(drop=True)

# Actual future realized volatility (recomputed from price in the dashboard)
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))
df['actual_volatility'] = df['log_ret'].rolling(VOL_WINDOW).std().shift(-VOL_WINDOW)

plot = df.dropna(subset=['actual_volatility', 'predicted_volatility']).iloc[:N_PLOT]

# RMSE for the title
rmse = np.sqrt(np.mean((plot['actual_volatility'] - plot['predicted_volatility'])**2))

plt.figure(figsize=(13, 7))
plt.plot(plot['actual_volatility'].values,    label='Actual Volatility',    color='blue', linewidth=1.0)
plt.plot(plot['predicted_volatility'].values, label='Predicted Volatility', color='red',  linewidth=1.0)
plt.title(f'{SYMBOL} LSTM Prediction Performance (Volatility)\nRMSE: {rmse:.5f}')
plt.xlabel('Time Steps (5m intervals)'); plt.ylabel('Realized Volatility')
plt.legend(loc='upper left'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  LOG RETURN PLOT  (standalone - run after the trainer cell)
#  Shows the actual log returns over the same test window.
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SYMBOL = 'XRPUSDT'
SEQ_LENGTH = 60
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1
N_PLOT = 500

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    BASE_DIR = os.path.abspath(os.getcwd())

CSV_PATH = os.path.join(BASE_DIR, 'data', f'{SYMBOL}_5m_data.csv')

# Load price data and compute log returns
df = pd.read_csv(CSV_PATH)
df['open_time'] = pd.to_datetime(df['open_time'])
df = df.sort_values('open_time').reset_index(drop=True)
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))

# Align to the SAME test window used by the trainer
n_rows     = len(df)
val_end    = int(n_rows * (TRAIN_SPLIT + VAL_SPLIT))
test_start = val_end + SEQ_LENGTH
log_ret_test = df['log_ret'].iloc[test_start : test_start + N_PLOT].values

# --- Plot ---
plt.figure(figsize=(14, 5))
plt.plot(log_ret_test, label='Log Return', color='purple', alpha=0.8, linewidth=1.0)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.title(f'{SYMBOL} Actual Log Returns (test window)')
plt.xlabel('Time Steps (5m)'); plt.ylabel('Log Return')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print(f"Log return stats (test window): mean={log_ret_test.mean():.6f}, std={log_ret_test.std():.6f}, "
      f"min={log_ret_test.min():.6f}, max={log_ret_test.max():.6f}")
